# 1. Carga del universo, variables derivadas y limpieza

**Objetivo.** Construir el *universo* de trabajo, es decir, todos los viajes válidos del dataset, a partir del cual se extraerá la muestra del equipo.

**Procedimiento.**
1. **Configuración:** `sys.path.append("..")` permite importar `src/utils.py` desde `notebooks/`; `%autoreload 2` recarga los cambios de `utils.py` sin reiniciar el kernel.
2. **Lectura:** se carga `train.csv` completo (`raw`) y se interpreta `pickup_datetime` como fecha.
3. **Variables derivadas** (`add_features`): distancia geodésica de Haversine, hora del día, día de la semana, indicador de fin de semana y velocidad promedio (distancia / duración).
4. **Limpieza** (`clean_trips`): se aplican cuatro reglas en secuencia y se reporta cuántas filas elimina cada una.

| Regla | Justificación |
|---|---|
| Coordenadas dentro de NYC (lat 40.5–41.0, lon −74.3 a −73.6) | Fuera de la ciudad son errores de GPS |
| Duración entre 1 min y 3 h | Menos de 1 min es cancelación o error; más de 3 h no es un viaje típico |
| Distancia ≥ 0.1 km | Un viaje sin desplazamiento no describe movilidad |
| Velocidad entre 1 y 100 km/h | Fuera de ese rango es físicamente implausible en la ciudad |

5. **Variables para el modelo:** se transforman a $\log d$ y $\log t$ (distancia y duración son asimétricas a la derecha) y la hora se codifica como $(\sin(2\pi h/24), \cos(2\pi h/24))$ para respetar su carácter circular (23:00 y 00:00 son cercanas).

**Resultado.** `universe` (viajes válidos), `FEATURES` (variables del modelo) y `N_UNIVERSE` (tamaño del universo).

In [2]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, entropy
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

from src.utils import DEFAULT_TRAIN, add_features, clean_trips

%load_ext autoreload
%autoreload 2

raw = pd.read_csv(DEFAULT_TRAIN, parse_dates=["pickup_datetime"])
print("Raw records:", len(raw))

universe = clean_trips(add_features(raw))     # prints rows removed per rule
universe["log_distance"] = np.log(universe["distance_km"])
universe["log_duration"] = np.log(universe["trip_duration"])
universe["hour_sin"] = np.sin(2 * np.pi * universe["hour"] / 24)
universe["hour_cos"] = np.cos(2 * np.pi * universe["hour"] / 24)

FEATURES = ["log_distance", "log_duration", "hour_sin", "hour_cos"]
N_UNIVERSE = len(universe)

Raw records: 1458644
inside NYC bounding box             removed    633  (left 1458011)
duration between 1 min and 3 h      removed  10595  (left 1447416)
distance >= 0.1 km                  removed   7463  (left 1439953)
speed between 1 and 100 km/h        removed   1342  (left 1438611)
Total kept: 1438611/1458644 (98.6%)


**Resultado de la limpieza.** De los 1 458 644 registros originales se conservan **1 438 611 (98.6 %)**; se eliminan 20 033 (1.4 %).

| Regla | Filas removidas | % del total original |
|---|---|---|
| Coordenadas fuera de NYC | 633 | 0.04 % |
| Duración fuera de 1 min – 3 h | 10 595 | 0.73 % |
| Distancia < 0.1 km | 7 463 | 0.51 % |
| Velocidad fuera de 1–100 km/h | 1 342 | 0.09 % |

**Interpretación.** La limpieza es leve: el mayor descarte proviene de duraciones atípicas y no altera de forma material la población. Las reglas se aplican en secuencia, así que cada viaje se cuenta en la primera regla que incumple. Los umbrales son decisiones metodológicas del equipo.

## Codificación cíclica de la hora del día

**El problema.** La hora (0–23) es una variable *circular*: después de las 23:00 vuelve a empezar en las 00:00. Sin embargo, como número, el modelo la trata como una recta con dos extremos, no como un reloj. Esto produce una distorsión en el borde:

$$|23 - 0| = 23 \quad \text{(el modelo "cree" que están muy lejos)}$$

cuando en realidad las 23:00 y las 00:00 están separadas por **una sola hora**.

**Por qué importa aquí.** El GMM agrupa los datos usando distancias (asume nubes gaussianas alrededor de un centro). Si la hora se deja como un número simple, el modelo puede separar artificialmente los viajes de las 23h y de la 1h en perfiles distintos, aunque en la realidad sean el mismo tipo de viaje nocturno.

**La solución: codificación cíclica.** Se transforma la hora en un ángulo sobre un círculo unitario, de modo que el ciclo no tenga extremos:

$$\theta = \frac{2\pi h}{24}, \qquad (\text{hour\_cos}, \text{hour\_sin}) = (\cos\theta, \sin\theta)$$

Con esta transformación, las 23:00 y las 00:00 quedan geométricamente cerca:

| Hora | hour_cos | hour_sin |
|---|---|---|
| 23:00 | 0.966 | −0.259 |
| 00:00 | 1.000 | 0.000 |

$$\text{distancia} = \sqrt{(0.966-1.000)^2 + (-0.259-0)^2} \approx 0.26 \quad \text{(cercana, correcto)}$$

**Por qué se usan ambas funciones y no solo una.** El seno por sí solo confunde horas distintas con el mismo valor (por ejemplo, 3:00 y 9:00 comparten $\sin\theta = 0.707$); el par (coseno, seno) identifica cada hora de forma única sobre el círculo.

**Nota técnica.** Aunque $\sin^2\theta + \cos^2\theta = 1$, esta relación es no lineal y no genera problemas de colinealidad en la matriz de covarianza del GMM (a diferencia de incluir, por ejemplo, $\log d$, $\log t$ y $\log v$ juntos, que sí sería una dependencia lineal exacta).

Esta técnica se conoce como **codificación cíclica** (*cyclical encoding*) y es estándar en ML para cualquier variable periódica: hora del día, día de la semana, mes del año, dirección del viento, etc.

**Limitación honesta.** Los puntos codificados así viven sobre un anillo, no forman una nube gaussiana "rellena". El GMM puede aproximar esta forma con varias componentes elípticas a lo largo del arco, y esto se debe tener en cuenta al interpretar los perfiles latentes.

In [3]:
universe.info()

<class 'pandas.DataFrame'>
RangeIndex: 1438611 entries, 0 to 1438610
Data columns (total 20 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   id                  1438611 non-null  str           
 1   vendor_id           1438611 non-null  int64         
 2   pickup_datetime     1438611 non-null  datetime64[us]
 3   dropoff_datetime    1438611 non-null  str           
 4   passenger_count     1438611 non-null  int64         
 5   pickup_longitude    1438611 non-null  float64       
 6   pickup_latitude     1438611 non-null  float64       
 7   dropoff_longitude   1438611 non-null  float64       
 8   dropoff_latitude    1438611 non-null  float64       
 9   store_and_fwd_flag  1438611 non-null  str           
 10  trip_duration       1438611 non-null  int64         
 11  distance_km         1438611 non-null  float64       
 12  hour                1438611 non-null  float64       
 13  dayofweek           143

# 2a. Justificación del tamaño de muestra — precisión estadística

**Pregunta.** ¿Qué tan confiables son las proporciones (por ejemplo, la fracción de viajes en una zona u hora) calculadas sobre una muestra de tamaño $n$?

**Fórmula.** Para una proporción $p$ estimada con $n$ observaciones, el error estándar es

$$SE = \sqrt{\frac{p(1-p)}{n}}$$

y el margen de error al 95% de confianza es $1.96 \cdot SE$, expresado en puntos porcentuales.

**Por qué importa.** El Rol 1 calcula entropía e información mutua a partir de proporciones de zonas y horas. Si $n$ es muy pequeño, esas proporciones (sobre todo las de zonas poco frecuentadas) son ruido y no reflejan la demanda real.

**Caso crítico.** El peor escenario para el margen absoluto es $p=0.50$; el escenario más informativo para justificar $n$ es una categoría rara (por ejemplo $p=0.01$), donde se ve si el muestreo alcanza para medirla con precisión.

In [6]:
def margin_pp(p, n):
    """Margen de error al 95%, en puntos porcentuales."""
    return 1.96 * np.sqrt(p * (1 - p) / n) * 100

rows = []
for n in [10_000, 20_000, 30_000, 40_000, 50_000]:
    rows.append({
        "n": n,
        "p=0.50 (zona común)": margin_pp(0.50, n),
        "p=0.05 (zona media)": margin_pp(0.05, n),
        "p=0.01 (zona rara)": margin_pp(0.01, n),
    })

precision_table = pd.DataFrame(rows).round(3)
precision_table

,n,p=0.50 (zona común),p=0.05 (zona media),p=0.01 (zona rara)
0,10000,0.980,0.427,0.195
1,20000,0.693,0.302,0.138
2,30000,0.566,0.247,0.113
3,40000,0.490,0.214,0.098
4,50000,0.438,0.191,0.087


**Resultado.** Con $n = 40\,000$:

| Escenario | Margen de error (95%) |
|---|---|
| Zona común ($p=0.50$) | ±0.490 puntos porcentuales |
| Zona media ($p=0.05$) | ±0.214 puntos porcentuales |
| Zona rara ($p=0.01$) | ±0.098 puntos porcentuales |

Incluso en el caso más sensible (una zona que representa apenas el 1% de la demanda), el margen de error es de una décima de punto porcentual, es decir, un error relativo de aproximadamente el 10% sobre ese 1%. Este nivel de precisión es suficiente para que los cálculos de entropía e información mutua del Rol 1 reflejen la estructura real de los datos y no ruido de muestreo.

**Rendimientos decrecientes.** Al comparar $n=10\,000$ con $n=40\,000$, el margen para una zona rara mejora de 0.195 a 0.098 puntos (se reduce a la mitad). Sin embargo, duplicar de $n=20\,000$ a $n=40\,000$ solo mejora de 0.138 a 0.098 (una reducción del 29%), y pasar a $n=50\,000$ apenas mejora a 0.087 (una ganancia adicional del 11%, con 25% más de datos). Esto refleja la relación $SE \propto 1/\sqrt{n}$: para reducir el margen a la mitad hay que **cuadruplicar** $n$, no duplicarlo. $n=40\,000$ está en la zona donde el margen ya es pequeño y cada dato adicional aporta cada vez menos precisión.

Bloque 1: Lo que falta, en orden
Sección	Qué hace	Por qué es necesaria
2b	Observaciones por parámetro del GMM	Justifica N desde tu propio modelo (Rol 3)
2c	Representatividad de la muestra vs. el universo	Prueba que la muestra no está sesgada
2d	Curva de estabilidad (log-verosimilitud vs. n)	Evidencia empírica de que N ya es suficiente
3	Decisión final, muestreo y guardado	Fija el N_SAMPLE y crea data/processed/sample.csv
4	Resumen para el PDF	Tabla final de todos los números que sustentan la decisión
Exploración básica (EDA)	Ver distribuciones antes de modelar	Detecta problemas que la limpieza no cubrió